# Internal Assessment 2 — Big Data / Machine Learning Dataset Assignment

**Student:** B A UdayKiran  
**USN:** 1AM23CI020  
**Department:** CSE-AIML  
**Semester:** VII  
**Subject:** Machine Learning / Big Data-related IA  

This notebook trains the three datasets supplied by the faculty:
1. Classification — `Social_Network_Ads.csv`
2. Boosting — `insurance_pre.csv`
3. Clustering — `Mall_Customers.csv`


## 1. Classification — Random Forest

Dataset: Social Network Ads

Target: `Purchased`

The `Gender` feature is one-hot encoded. `Age` and `EstimatedSalary` are used as numerical features. A Random Forest classifier is trained and evaluated using accuracy, confusion matrix, precision, recall and F1-score.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("../datasets/Social_Network_Ads.csv")
X = df[["Gender", "Age", "EstimatedSalary"]]
y = df["Purchased"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["Gender"]),
    ("num", "passthrough", ["Age", "EstimatedSalary"])
])

model = Pipeline([
    ("preprocessor", pre),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


## 2. Boosting — AdaBoost Regression

Dataset: `insurance_pre.csv`

Target: `charges`

Because `charges` is a continuous numerical value, this dataset is treated as a **regression** problem. AdaBoost is therefore used with a decision-tree regressor as the weak learner.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../datasets/insurance_pre.csv")
X = df.drop(columns=["charges"])
y = df["charges"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["sex", "smoker"]),
    ("num", "passthrough", ["age", "bmi", "children"])
])

model = Pipeline([
    ("preprocessor", pre),
    ("regressor", AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4, random_state=42),
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)


## 3. Clustering — K-Means

Dataset: `Mall_Customers.csv`

Features used:
- Annual Income (k$)
- Spending Score (1-100)

The data is standardized before K-Means. The silhouette score is checked for k = 2 to 10, and the best k is selected from the highest silhouette score.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../datasets/Mall_Customers.csv")
features = df[["Annual Income (k$)", "Spending Score (1-100)"]]

scaler = StandardScaler()
X = scaler.fit_transform(features)

scores = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Silhouette scores:", scores)
print("Best k:", best_k)

km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(X)
print("Final silhouette score:", silhouette_score(X, labels))

plt.figure(figsize=(7,5))
for lab in sorted(set(labels)):
    pts = X[labels == lab]
    plt.scatter(pts[:,0], pts[:,1], label=f"Cluster {lab+1}", alpha=0.7)
centers = km.cluster_centers_
plt.scatter(centers[:,0], centers[:,1], marker="X", s=160, label="Centroids")
plt.title(f"K-Means Clustering (k={best_k})")
plt.xlabel("Standardized Annual Income")
plt.ylabel("Standardized Spending Score")
plt.legend()
plt.show()


## Final Result

The notebook demonstrates three different machine-learning tasks on the exact datasets supplied for the IA:
- Random Forest classification on Social Network Ads
- AdaBoost regression on Insurance data
- K-Means clustering on Mall Customers

The generated results and graphs are stored in the `results` folder.
